<a href="https://colab.research.google.com/github/23f1002879/Pytorch-Computer-Vision/blob/master/notebooks/05_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torchvision import models

In [2]:
weights =  models.ResNet18_Weights.DEFAULT # Inititating the RESNET18 weights

model = models.resnet18(weights=weights)  # Initiating the resnet18 architecture
print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 202MB/s]

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [3]:
# we don't want to retain the millions of parameter from the ResNet18 Since our FashionMNIST had only 10 classes
for param in model.parameters():
    param.requires_grad = False


In [4]:
# Replacing the final Layer of the ResNet_18 Architecture
model.fc = nn.Linear(
    in_features=512,
    out_features=10
)

for name, param in model.named_parameters():

    if param.requires_grad:
        print(name)

fc.weight
fc.bias


In [5]:
''' Since for our case the FashionMNIST data consist of only 1 channel and 28x28 Input features But ResNet_18 expect 3 channels 224x224 features. Therefore, there is need to transform the data '''


from torchvision import transforms, datasets

transform = transforms.Compose([
    transforms.Resize((
        224,
        224
    )),
    transforms.Grayscale(
        num_output_channels=3
    ),
    transforms.ToTensor()
])

In [6]:
trainDataset = datasets.FashionMNIST(
    root='data',
    train=True,
    download=True,
    transform=transform
)

testDataset = datasets.FashionMNIST(
    root='data',
    train=False,
    download=True,
    transform=transform
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 173kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.25MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.1MB/s]


In [7]:
from torch.utils.data import DataLoader

trainLoader = DataLoader(
    trainDataset,
    batch_size=64,
    shuffle=True
)

testLoader = DataLoader(
    testDataset,
    batch_size=64,
    shuffle=True
)

In [8]:
device = torch.device(
'cuda' if torch.cuda.is_available() else 'cpu'
)

model=model.to(device)

In [9]:
# Setting Up the Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
epoch = int(input("Enter the number of epochs:"))

for i in range(epoch):
  for image, label in trainLoader:
      image, label = image.to(device), label.to(device)
      output = model(image)
      loss = criterion(output, label)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
  print(loss.item())

Enter the number of epochs:3
0.3593760132789612
